# Khám phá dữ liệu

## Import thư viện

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
import time
from requests.exceptions import Timeout, RequestException
import requests
import re
import os
from typing import Dict
from tqdm import tqdm

## Load data từ driver

In [ ]:
drive.mount('/content/drive')

In [ ]:
csv_path = '/content/drive/MyDrive/job_descriptions.csv'
usecols = ['Job Id', 'Job Title', 'Role', 'Experience', 'Qualifications', 'Job Description', 'skills', 'Responsibilities', 'Company']
df_jd = pd.read_csv(csv_path, usecols=usecols)
print(f"Tải thành công! Kích thước dữ liệu: {df_jd.shape[0]} dòng, {df_jd.shape[1]} cột.")

In [ ]:
display(df_jd.head())

In [ ]:
df_jd.info()

In [ ]:
print(df_jd.isnull().sum())

In [ ]:
duplicates = df_jd.duplicated().sum()
print(f"\nSố lượng bản ghi trùng lặp: {duplicates}")

In [ ]:
# Tạo thêm cột độ dài từ (Word Count) cho Job Description
df_jd['JD_Word_Count'] = df_jd['Job Description'].apply(lambda x: len(str(x).split()))

In [ ]:
# Vẽ biểu đồ phân bố độ dài văn bản
plt.figure(figsize=(10, 6))
sns.histplot(df_jd['JD_Word_Count'], bins=50, color='blue', kde=True)
plt.title('Phân bố độ dài của Job Description (Số lượng từ)')
plt.xlabel('Số từ')
plt.ylabel('Tần suất')
plt.axvline(x=df_jd['JD_Word_Count'].mean(), color='red', linestyle='--', label='Trung bình')
plt.legend()
plt.show()

In [ ]:
# Lọc bỏ các JD quá ngắn (dưới 10 từ) hoặc quá dài (trên 1000 từ) theo tiêu chuẩn bài báo 2022
df_jd = df_jd[(df_jd['JD_Word_Count'] >= 10) & (df_jd['JD_Word_Count'] <= 1000)]
print(f"Số lượng JD hợp lệ sau khi lọc độ dài: {df_jd.shape}")

In [ ]:
print(df_jd['Role'].value_counts().head(15))

In [ ]:
SAMPLE_SIZE = 10000

top_roles = df_jd['Role'].value_counts().nlargest(20).index
df_filtered = df_jd[df_jd['Role'].isin(top_roles)]

df_sampled = df_filtered.sample(n=SAMPLE_SIZE, random_state=42)

print(f"Đã lấy mẫu thành công: {df_sampled.shape} dòng.")

In [ ]:
df_sampled = pd.read_csv('/content/drive/MyDrive/jd_sample_data.csv')
df_sampled.to_csv('/content/drive/MyDrive/jd_sample_data.csv', index=False)

In [ ]:
input_path = '/content/drive/MyDrive/jd_sample_data.csv'
df_jd = pd.read_csv(input_path)

df_structured_jd = pd.DataFrame()

df_structured_jd['Job_ID'] = df_jd['Job Id']
df_structured_jd['TITLE'] = df_jd['Job Title']
df_structured_jd['DESCRIPTION'] = df_jd['Job Description']
df_structured_jd['RESPONSIBILITIES'] = df_jd['Responsibilities']
df_structured_jd['EDUCATION'] = df_jd['Qualifications']
df_structured_jd['SKILLS'] = df_jd['skills']
df_structured_jd['EXPERIENCE'] = df_jd['Experience']
df_structured_jd['ADDITIONAL_INFO'] = (
    ". Role: " + df_jd['Role'].astype(str) +
    ". Company Profile: " + df_jd['Company'].astype(str)
)

df_structured_jd = df_structured_jd.fillna("")

output_path = '/content/drive/MyDrive/JD_Structured_Master.csv'
df_structured_jd.to_csv(output_path, index=False)

print(f"ĐÃ ÁNH XẠ XONG! File JD Master có kích thước: {df_structured_jd.shape}")
display(df_structured_jd.head(3))

In [ ]:
file_path = '/content/drive/MyDrive/JD_Structured_Master.csv'
df_jd = pd.read_csv(file_path)

edu_mapping = {
    'BA': "Bachelor of Arts degree",
    'BBA': "Bachelor of Business Administration degree",
    'BCA': "Bachelor of Computer Applications degree",
    'B.Com': "Bachelor of Commerce degree",
    'B.Tech': "Bachelor of Technology degree",
    'MBA': "Master of Business Administration degree",
    'MCA': "Master of Computer Applications degree",
    'M.Com': "Master of Commerce degree",
    'M.Tech': "Master of Technology degree",
    'PhD': "Doctor of Philosophy (PhD) degree"
}

def expand_education(edu_abbr):
    abbr = str(edu_abbr).strip()
    full_name = edu_mapping.get(abbr, abbr)

    return f"A {full_name} is required or highly preferred for this position."

df_jd['EDUCATION'] = df_jd['EDUCATION'].apply(expand_education)

df_jd.to_csv(file_path, index=False)
print("✅ Đã chuẩn hóa và mở rộng thành công cột EDUCATION!")
display(df_jd.head(5))

In [ ]:
import os
import pandas as pd
!pip install PyPDF2
import PyPDF2
import re
import requests
from enum import Enum
from typing import Dict

#  AppConfig

In [ ]:
class DocumentPart(Enum):
    TITLE = "TITLE"
    DESCRIPTION = "DESCRIPTION"
    RESPONSIBILITIES = "RESPONSIBILITIES"
    EDUCATION = "EDUCATION"
    SKILLS = "SKILLS"
    EXPERIENCE = "EXPERIENCE"
    ADDITIONAL_INFO = "ADDITIONAL_INFO"

class Config:
    MISTRAL_API_KEY = "vwNrm5KXc7X7D9nHzKche6ByJ7oL6WdI"
    MISTRAL_URL = "https://api.mistral.ai/v1/chat/completions"

CV_FOLDER_PATH = '/content/drive/MyDrive/CV_Dataset'
MASTER_CSV_PATH = '/content/drive/MyDrive/CV_Structured_Master.csv' # File lưu kết quả đã bóc tách
TRACKING_FILE = '/content/drive/MyDrive/processed_cvs_llm.txt'
DATA_CV_DIR = '/content/drive/MyDrive/DATA_cv'

# LLM

In [ ]:
class PromptService:
    @staticmethod
    def enhance_and_structure_resume(resume_text: str) -> str:
        # Prompt Chain-of-Thought
        system_prompt = (
            "You are an AI writing assistant who does resume editing. Your task is to create additional content IF NEEDED to a resume and structure it.\n"
            "When given a resume you have to structure the content per categories: TITLE, DESCRIPTION, RESPONSIBILITIES, EDUCATION, SKILLS, EXPERIENCE, ADDITIONAL_INFO.\n"
            "If content is missing fill with minimum requirements. Do not create new category or any inner list, "
            "Any information that do not match put it to the ADDITIONAL_INFO category. The categories must not be empty.\n"
            "You must put placeholders of ^-^ symbol between categories and descriptions.\n"
            "EXAMPLE:\n"
            "^-^TITLE^-^Software Engineer^-^\n"
            "^-^DESCRIPTION^-^Design, develop, and maintain software applications...^-^\n"
            "^-^RESPONSIBILITIES^-^Collaborate with a team of software developers...^-^\n"
            "^-^EDUCATION^-^Bachelor's degree in Computer Science...^-^\n"
            "^-^SKILLS^-^Strong programming skills in Java, Python...^-^\n"
            "^-^EXPERIENCE^-^At least 2-3 years of experience...^-^\n"
            "^-^ADDITIONAL_INFO^-^Speak fluently English...^-^\n"
            "Do not include any explanations, follow this format without deviation."
        )
        return f"{system_prompt}\n\nRewrite AND structure per categories the resume below: {resume_text}"


class ParseService:
    @staticmethod
    def call_mistral(prompt: str, max_retries=3) -> str:
        headers = {
            "Authorization": f"Bearer {Config.MISTRAL_API_KEY}",
            "Content-Type": "application/json"
        }
        data = {
            "model": "open-mistral-7b",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.7
        }

        for attempt in range(max_retries):
            try:
                response = requests.post(Config.MISTRAL_URL, headers=headers, json=data, timeout=45)

                if response.status_code == 200:
                    return response.json()['choices'][0]['message']['content']
                elif response.status_code == 429:
                    print(f"Quá tải API (Rate Limit)")
                else:
                    print(f"Lỗi Mistral ({response.status_code}): {response.text}")

            except Timeout:
                print(f"API phản hồi quá chậm (Timeout) ở lần thử thứ {attempt + 1}.")
            except RequestException as e:
                print(f"Lỗi kết nối mạng: {e}")

            time.sleep(2 ** attempt)

        raise Exception("API Mistral thất bại sau tất cả các lần thử.")

    @staticmethod
    def extract_text_parts(llm_output: str) -> Dict[str, str]:
        # Step 4 & 6: Tách chuỗi theo schema ^-^ và làm sạch (Data cleaning)
        parts = llm_output.split('^-^')
        mapped_parts = {}
        valid_keys = [e.value for e in DocumentPart]

        for i in range(len(parts) - 1):
            key_candidate = parts[i].strip()
            if key_candidate in valid_keys:
                raw_text = parts[i + 1].strip()
                clean_text = re.sub(r'[^a-zA-Z0-9.]', ' ', raw_text)
                clean_text = re.sub(r'\s{2,}', ' ', clean_text).strip()
                mapped_parts[key_candidate] = clean_text
        return mapped_parts

    @staticmethod
    def extract_category(llm_output: str) -> str:
        match = re.search(r'\^-\^CATEGORY\^-\^(.*?)\^-\^', llm_output, re.IGNORECASE | re.DOTALL)
        if match:
            return match.group(1).strip().lower()
        return "unknown"

# QUẢN LÝ TIẾN TRÌNH & ĐỌC PDF

In [ ]:
def load_processed_files():
    if os.path.exists(TRACKING_FILE):
        with open(TRACKING_FILE, 'r') as f:
            return set(f.read().splitlines())
    return set()

def save_processed_file(filename):
    with open(TRACKING_FILE, 'a') as f:
        f.write(filename + '\n')

def extract_text_from_pdf(file_path):
    text = ""
    try:
        with open(file_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            for page in pdf_reader.pages:
                extracted = page.extract_text()
                if extracted:
                    text += extracted + " "
        return re.sub(r'\s+', ' ', text).strip()
    except Exception as e:
        print(f"[Lỗi] Không đọc được file {file_path}: {e}")
        return None

# ORCHESTRATOR - TÍCH HỢP LLM

In [ ]:
def run_llm_incremental_cv_loader():
    processed_files = load_processed_files()
    if not os.path.exists(CV_FOLDER_PATH):
        print(f"Không tìm thấy thư mục {CV_FOLDER_PATH}.")
        return

    all_pdf_files = [f for f in os.listdir(CV_FOLDER_PATH) if f.endswith('.pdf')]

    print(f"Tổng CV trong thư mục: {len(all_pdf_files)} | Đã tách bởi LLM trước đó: {len(processed_files)}")

    for filename in all_pdf_files:
        if filename not in processed_files:
            file_path = os.path.join(CV_FOLDER_PATH, filename)
            print(f"Đang xử lý bằng Mistral-7B cho CV: {filename}...")

            # Step 1: Đọc PDF
            raw_text = extract_text_from_pdf(file_path)

            if raw_text:
                try:
                    # Step 2 & 3: Tạo Prompt và gọi Mistral LLM
                    prompt = PromptService.enhance_and_structure_resume(raw_text)
                    llm_response = ParseService.call_mistral(prompt)

                    # Step 4 & 6: tách thành 7 trường
                    structured_data = ParseService.extract_text_parts(llm_response)
                    structured_data['Resume_ID'] = filename

                    df_single = pd.DataFrame([structured_data])

                    if os.path.exists(MASTER_CSV_PATH):
                        df_single.to_csv(MASTER_CSV_PATH, mode='a', header=False, index=False)
                    else:
                        df_single.to_csv(MASTER_CSV_PATH, index=False)

                    save_processed_file(filename)

                except Exception as e:
                    print(f"-> [Cảnh báo] Lỗi API Mistral cho {filename}: {e}")

    print("==> HỆ THỐNG ĐÃ QUÉT XONG TOÀN BỘ THƯ MỤC!")

if __name__ == "__main__":
    run_llm_incremental_cv_loader()

# Add new Category colunm into the

In [ ]:
df_cv = pd.read_csv(MASTER_CSV_PATH)

file_to_category = {}
for category_folder in os.listdir(DATA_CV_DIR):
    folder_path = os.path.join(DATA_CV_DIR, category_folder)
    if os.path.isdir(folder_path):
        for filename in os.listdir(folder_path):
            if filename.endswith('.pdf'):
                file_to_category[filename] = category_folder.strip().upper()

df_cv['Category'] = df_cv['Resume_ID'].map(file_to_category).fillna('UNKNOWN')

# Lưu lại đè lên file cũ
df_cv.to_csv(MASTER_CSV_PATH, index=False)
print("Đã cập nhật thành công cột Category cho tập CV!")
display(df_cv[['Resume_ID', 'Category']].head())

# Cosine similarity

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from numpy.linalg import norm
import random
import time
import os
from huggingface_hub import login


In [ ]:
CV_PATH = '/content/drive/MyDrive/CV_Structured_Master.csv'
JD_PATH = '/content/drive/MyDrive/JD_Structured_Master.csv'
OUTPUT_DATASET_PATH = '/content/drive/MyDrive/Labeled_Dataset_for_GNN.csv'

CORE_FIELDS = ['TITLE', 'DESCRIPTION', 'RESPONSIBILITIES', 'SKILLS', 'EXPERIENCE']


# Dùng Gemma 300M
model = SentenceTransformer('google/embeddinggemma-300m')

## EMBEDDING CACHING

In [ ]:
def compute_embeddings(df, id_col):
    """Tính vector cho từng trường của từng file và lưu vào bộ nhớ đệm (Dictionary)"""
    embedding_cache = {}

    for _, row in df.iterrows():
        record_id = row[id_col]
        vectors = {}
        for field in CORE_FIELDS:
            text = str(row.get(field, ""))
            if text.strip() == "":
                # Nếu rỗng, tạo vector 0
                vectors[field] = np.zeros(model.get_embedding_dimension())
            else:
                vectors[field] = model.encode(text)
        embedding_cache[record_id] = vectors
    return embedding_cache

def calculate_cosine(vec_a, vec_b):
    if norm(vec_a) == 0 or norm(vec_b) == 0:
        return 0.0
    return np.dot(vec_a, vec_b) / (norm(vec_a) * norm(vec_b))

In [ ]:
df_cv = pd.read_csv(CV_PATH).fillna("")
df_jd = pd.read_csv(JD_PATH).fillna("")

cv_ids = df_cv['Resume_ID'].tolist()
jd_ids = df_jd['Job_ID'].tolist()

print(f"Đã tải {len(cv_ids)} CVs và {len(jd_ids)} JDs.")

print("\n--- TẠO VECTOR CHO TOÀN BỘ DATA ---")
start_time = time.time()
print("-> Đang mã hóa CVs...")
cv_embeddings = compute_embeddings(df_cv, 'Resume_ID')
print("-> Đang mã hóa JDs...")
jd_embeddings = compute_embeddings(df_jd, 'Job_ID')
print(f"Hoàn tất mã hóa trong {round(time.time() - start_time, 2)} giây!")

processed_jds = set()
if os.path.exists(OUTPUT_DATASET_PATH):
    try:
        df_existing = pd.read_csv(OUTPUT_DATASET_PATH)
        processed_jds = set(df_existing['Job_ID'].unique())
    except pd.errors.EmptyDataError:
        pass

print(f"-> Đã có {len(processed_jds)} JD được xử lý từ trước. Bỏ qua!")

for jd_id, jd_row in df_jd.iterrows():
    real_jd_id = jd_row['Job_ID']

    if real_jd_id in processed_jds:
        continue

     # 1. Lấy dữ liệu từ cột TITLE
    jd_title = str(jd_row.get('TITLE', '')).strip().upper()

    # 2. Trích xuất ROLE từ cột ADDITIONAL_INFO bằng Regex
    additional_info = str(jd_row.get('ADDITIONAL_INFO', ''))

    # Hàm re.search sẽ tìm nội dung nằm giữa ". Role: " và ". Company Profile:"
    # (Dùng re.IGNORECASE để không phân biệt hoa thường chữ Role và Company Profile)
    role_match = re.search(r'\.\s*Role:\s*(.*?)\.\s*Company Profile:', additional_info, re.IGNORECASE)

    # Nếu tìm thấy, lấy group(1) (phần nằm trong ngoặc tròn), nếu không thì để trống
    jd_role = role_match.group(1).strip().upper() if role_match else ""

    # 3. LỌC CV THỰC TẾ (Lấy CV cùng ngành)
    relevant_cvs = df_cv[
      df_cv['Category'].apply(
        lambda cat: (cat in jd_title) or
          (jd_title in cat) or
          (jd_role != "" and cat in jd_role) or
          (jd_role != "" and jd_role in cat)
      )
    ]

    if relevant_cvs.empty:
        continue

    jd_vecs = jd_embeddings[real_jd_id]
    jd_scores_list = []

    # Tính điểm JD này với tất cả ứng viên (CV) nộp vào
    for _, cv_row in relevant_cvs.iterrows():
        cv_id = cv_row['Resume_ID']
        cv_vecs = cv_embeddings[cv_id]

        total_score = 0
        for field in CORE_FIELDS:
            total_score += calculate_cosine(jd_vecs[field], cv_vecs[field])

        overall_score = total_score / len(CORE_FIELDS)
        jd_scores_list.append((cv_id, overall_score))

    # XẾP HẠNG ỨNG VIÊN TỪ CAO XUỐNG THẤP (Ranking như bài báo 2025)
    jd_scores_list.sort(key=lambda x: x[1], reverse=True)

    dataset_records = []

    # GÁN NHÃN MÔ PHỎNG THỰC TẾ:
    # Những người điểm cao nhất (> 0.75) được mời phỏng vấn và trúng tuyển (Nhãn 1)
    passed_candidates = [item for item in jd_scores_list if item[1] >= 0.75]

    # Những người tuy cùng ngành, nộp CV nhưng kỹ năng/kinh nghiệm yếu hơn (<0.70) bị loại (Nhãn 0)
    failed_candidates = [item for item in jd_scores_list if item[1] < 0.70]

    if len(passed_candidates) > 0:
        # Lưu nhãn 1
        for cv_id, score in passed_candidates:
            dataset_records.append({'Resume_ID': cv_id, 'Job_ID': real_jd_id, 'Cosine_Score': round(score, 4), 'Label_Y': 1})

        for cv_id, score in failed_candidates:
            dataset_records.append({'Resume_ID': cv_id, 'Job_ID': real_jd_id, 'Cosine_Score': round(score, 4), 'Label_Y': 0})

    # Ghi xuống ổ cứng
    if dataset_records:
        df_single_jd = pd.DataFrame(dataset_records)
        if os.path.exists(OUTPUT_DATASET_PATH):
            df_single_jd.to_csv(OUTPUT_DATASET_PATH, mode='a', header=False, index=False)
        else:
            df_single_jd.to_csv(OUTPUT_DATASET_PATH, index=False)

print("==> xong")

In [ ]:
df_dataset = pd.read_csv(OUTPUT_DATASET_PATH)

print(df_dataset['Label_Y'].value_counts())

df_dataset.to_csv(OUTPUT_DATASET_PATH, index=False)
print(f"\nĐã lưu tập dữ liệu GNN tại: {OUTPUT_DATASET_PATH}")
display(df_dataset.head())